## BULU DOCUMENT PROCESSING BY CUSTOMIZED OCR PIPELINE
* BASED ON DEEPSEEK-OCR
* USING USLOTH LIBRARY
* SAT(sat-12l-sm) model (Frohmann et al. (2024)) for sentence level reconstitution/splitting

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install jiwer
!pip install einops addict easydict

### Unsloth

Let's prepare the OCR model to our local first

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download("unsloth/DeepSeek-OCR", local_dir = "deepseek_ocr")

In [ ]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch
from transformers import AutoModel
import os
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = '0'
# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Qwen3-VL-8B-Instruct-bnb-4bit", # Qwen 3 vision support
    "unsloth/Qwen3-VL-8B-Thinking-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Instruct-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Thinking-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastVisionModel.from_pretrained(
    "./deepseek_ocr",
    load_in_4bit = False, # Use 4bit to reduce memory use. False for 16bit LoRA.
    auto_model = AutoModel,
    trust_remote_code=True,
    unsloth_force_compile=True,
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.11.6: Fast Deepseekocr patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at ./deepseek_ocr and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 1. Let's Evaluate Deepseek-OCR Baseline Performance on Bulu Transcription set

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch

MODEL_NAME = "deepseek-ai/DeepSeek-OCR"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    _attn_implementation="flash_attention_2",
).eval().cuda()


In [ ]:
# prompt = "<image>\nFree OCR. "
prompt = "<image>\nFree OCR. "
image_file = 'bum-ocr_char.png'
output_path = 'output/dir'
# infer(self, tokenizer, prompt='', image_file='', output_path = ' ', base_size = 1024, image_size = 640, crop_mode = True, test_compress = False, save_results = False):

# Tiny: base_size = 512, image_size = 512, crop_mode = False
# Small: base_size = 640, image_size = 640, crop_mode = False
# Base: base_size = 1024, image_size = 1024, crop_mode = False
# Large: base_size = 1280, image_size = 1280, crop_mode = False

# Gundam: base_size = 1024, image_size = 640, crop_mode = True

res = model.infer(tokenizer, prompt=prompt, image_file=image_file, output_path = output_path, base_size = 1024, image_size = 640, crop_mode=True, save_results = True, test_compress = False)


In [ ]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 102.8 MB/s eta 0:00:00


## 2. OCR PIPELINE: Scanned_doc_to_txt_file

In [ ]:

import fitz
from PIL import Image
from io import BytesIO

def pdf_page_to_image(pdf_path, page_index, dpi=300):
    doc = fitz.open(pdf_path)
    page = doc.load_page(page_index)
    zoom = dpi / 72
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    img = Image.open(BytesIO(pix.tobytes("png")))
    doc.close()
    return img


In [ ]:
from io import StringIO
import sys
import tempfile
import os

def run_deepseek(image, prompt, mode="Gundam"):
    config = {
        "Gundam": {"base_size": 1024, "image_size": 640, "crop_mode": True},
        "Tiny": {"base_size": 512, "image_size": 512, "crop_mode": False},
        "Small": {"base_size": 640, "image_size": 640, "crop_mode": False},
        "Base": {"base_size": 1024, "image_size": 1024, "crop_mode": False},
        "Large": {"base_size": 1280, "image_size": 1280, "crop_mode": False},
    }[mode]

    tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.jpg')
    image.save(tmp.name, "JPEG", quality=95)

    # capture STDOUT
    old_stdout = sys.stdout
    sys.stdout = StringIO()

    model.infer(
        tokenizer=tokenizer,
        prompt=prompt,
        image_file=tmp.name,
        output_path="/tmp",
        base_size=config["base_size"],
        image_size=config["image_size"],
        crop_mode=config["crop_mode"],
    )

    output = sys.stdout.getvalue()
    sys.stdout = old_stdout
    os.unlink(tmp.name)

    # Nettoyage pour enlever les logs internes
    clean = "\n".join([
        l for l in output.split("\n")
        if not any(k in l for k in ["PATCHES", "BASE:", "image:", "torch.Size", "===="])
    ])
    return clean.strip()


In [ ]:
def process_pdf(pdf_path, task="markdown", mode="Gundam"):
    doc = fitz.open(pdf_path)
    n_pages = len(doc)
    doc.close()

    final_text = ""

    # définir le prompt DeepSeek
    if task == "markdown":
        prompt = "<image>\n<|grounding|>Convert the document to markdown."
    elif task == "text":
        prompt = "<image>\nFree OCR."
    else:
        raise ValueError("task must be 'markdown' or 'text'")

    for i in range(n_pages):
        print(f"🔍 Processing page {i+1}/{n_pages}...")
        img = pdf_page_to_image(pdf_path, i)
        page_out = run_deepseek(img, prompt, mode)

        final_text += f"\n\n### --- PAGE {i+1} ---\n\n"
        final_text += page_out

    return final_text


### PIPELINE USAGE

In [ ]:
pdf_path = "/content/doc.pdf"

result = process_pdf(pdf_path, task="markdown", mode="Gundam")

with open("final_markdown.md", "w", encoding="utf-8") as f:
    f.write(result)

print("Document final généré → final_markdown.md")


PDF processing:   0%|          | 0/85 [00:00<?, ?it/s]

🔍 Processing page 1/85...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
PDF processing:   1%|          | 1/85 [00:34<47:36, 34.01s/it]

🔍 Processing page 2/85...
⏭️ Skipping page 2 (no processing)
🔍 Processing page 3/85...
⏭️ Skipping page 3 (no processing)
🔍 Processing page 4/85...
⏭️ Skipping page 4 (no processing)
🔍 Processing page 5/85...
⏭️ Skipping page 5 (no processing)
🔍 Processing page 6/85...
⏭️ Skipping page 6 (no processing)
🔍 Processing page 7/85...
⏭️ Skipping page 7 (no processing)
🔍 Processing page 8/85...
⏭️ Skipping page 8 (no processing)
🔍 Processing page 9/85...
⏭️ Skipping page 9 (no processing)
🔍 Processing page 10/85...
⏭️ Skipping page 10 (no processing)
🔍 Processing page 11/85...
⏭️ Skipping page 11 (no processing)
🔍 Processing page 12/85...
⏭️ Skipping page 12 (no processing)
🔍 Processing page 13/85...
⏭️ Skipping page 13 (no processing)
🔍 Processing page 14/85...
⏭️ Skipping page 14 (no processing)
🔍 Processing page 15/85...
⏭️ Skipping page 15 (no processing)
🔍 Processing page 16/85...
⏭️ Skipping page 16 (no processing)
🔍 Processing page 17/85...
⏭️ Skipping page 17 (no processing)
🔍 Proces

PDF processing:  24%|██▎       | 20/85 [01:16<03:39,  3.37s/it]

...

🔍 Processing page 84/85...


PDF processing:  99%|█████████▉| 84/85 [49:11<00:50, 50.45s/it]

🔍 Processing page 85/85...


PDF processing: 100%|██████████| 85/85 [49:52<00:00, 35.20s/it]

Document final généré → final doc is ready!


### 3. TEST SaT TOOL FOR SENTENCE LEVEL RECONSTITUTION/SPLITTING

In [ ]:
!pip install wtpsplit

In [ ]:
from wtpsplit import SaT

model = ['sat-3l','sat-12l-sm']

sat = SaT(model[1])
# optionally run on GPU for better performance
# also supports TPUs via e.g. sat.to("xla:0"), in that case pass `pad_last_batch=True` to sat.split
sat.half().to("cuda")


In [ ]:

input_file = "/content/FINAL_nnanga_kon.txt"
output_file = "/content/FINAL_Sent1_nnanga_kon.txt"



# 2. Charger tout le texte
with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()

# 3. Diviser ton texte en paragraphes/lignes pour éviter mémoire excessive
lines = [line.strip() for line in text.split("\n") if line.strip()]

# 4. Segmenter toutes les lignes en BATCH
sentences_batches = sat.split(lines)

# 5. Aplatir les résultats
sentences = []
for batch in sentences_batches:
    for s in batch:
        s = s.strip()
        if s:
            sentences.append(s)

# 6. Enregistrer les phrases segmentées
with open(output_file, "w", encoding="utf-8") as f:
    for s in sentences:
        f.write(s + "\n")

print("Segmentation terminée. Nombre de phrases :", len(sentences))


Segmentation terminée. Nombre de phrases : 1780


## option 2


In [ ]:
from wtpsplit import SaT

input_file = "/content/FINAL_nnanga_kon.txt"
output_file = "/content/FINAL_sent2_nnanga_kon.txt"


# 2. Charger le texte
with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()

# 3. Segmentation paragraphe + phrase
paragraphs = sat.split(
    text,
    do_paragraph_segmentation=True,
    paragraph_threshold=0.6  # ajustable
)

# 4. Aplatir toutes les phrases
all_sentences = []
for paragraph in paragraphs:
    for sentence in paragraph:
        sentence = sentence.strip()
        if sentence:
            all_sentences.append(sentence)

# 5. Sauvegarde
with open(output_file, "w", encoding="utf-8") as f:
    for sent in all_sentences:
        f.write(sent + "\n")

print("Nombre total de phrases :", len(all_sentences))
print("Nombre de paragraphes :", len(paragraphs))


Nombre total de phrases : 2285
Nombre de paragraphes : 79
